## 08 - Sanity Check: XDec Stage-2 TPM Sensitivity Analysis

### Aim
* Verify the integrity, numerical correctness, and reported results of the separate TPM-versus-raw-count Stage-2 sensitivity analysis.

### Input
* Saved run `data/deconvolution/sensitivity/stage2_tpm_20261031_122707/`, original expression matrices, Stage-1 proportions, tertile groups, and raw-count Stage-2 outputs.

### Output
* Executed validation results in this notebook: provenance checks, input/output sanity checks, independent NNLS comparisons, DE/FDR reproduction, and enrichment-status verification. Existing analysis files are read only.

### Steps
* Verify file hashes and sample/group alignment; check finite nonnegative estimates; independently solve 36 NNLS examples; recompute all DE tests and BH corrections; reproduce original nominal gene sets and summarize pathway overlap.

### Result
* See executed checks below. The saved comparison reports 0/37/50 raw-count versus 0/6/17 TPM genes for CAF/Lymphocyte/Enteroendocrine after BH FDR < 0.05 plus the fivefold filter; all 6 lymphocyte and 17 enteroendocrine TPM genes are shared with the corresponding raw-count lists.

### Conclusion
* These checks assess computational consistency and preservation of original files. They do not establish TPM as the uniquely correct input or validate the conditional z-test assumptions; the immune-related lymphocyte signal persists while exact gene lists depend on input units.


### 1. Locate the saved run and load dependencies

Paths are resolved relative to the `progress/` pipeline root. This notebook does not run XDec again or modify existing analysis outputs. All checks use assertions: an inconsistency stops execution.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import numpy as np
import pandas as pd
from scipy.optimize import nnls
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

candidates = [Path.cwd(), *Path.cwd().parents]
PROGRESS = next((p for parent in candidates for p in (parent, parent / "progress")
                 if (p / "workflow/Snakefile").is_file()), None)
assert PROGRESS is not None, "Run from this repository or its notebook directory"
ROOT = PROGRESS.parent
RUN = PROGRESS / "data/deconvolution/sensitivity/stage2_tpm_20261031_122707"
ORIGINAL = PROGRESS / "data/deconvolution/xdec"
PROFILES = ["Profile_2", "Profile_5", "Profile_8"]
checks = []
def passed(name):
    checks.append(name)
    print("PASS:", name)
print("Run:", RUN)
print("Python:", sys.executable)
print("Platform:", platform.platform())
manifest = json.loads((RUN / "manifest.json").read_text())
assert manifest["status"] == "complete"
passed("Saved sensitivity analysis is complete")

Run: /Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer/progress/data/deconvolution/sensitivity/stage2_tpm_20261031_122707
Python: /Users/liruixi/.pixi/envs/python/bin/python
Platform: macOS-15.3.2-arm64-arm-64bit
PASS: Saved sensitivity analysis is complete


### 2. Verify provenance and preservation of existing files

Recalculate SHA-256 hashes for the explicit inputs, all original XDec artifacts captured before the comparison, and every sensitivity output in the saved integrity manifest. These checks detect changes after the analysis; they cannot establish biological validity.

In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(2**20), b""):
            digest.update(chunk)
    return digest.hexdigest()

for label, hashes, base in [
    ("explicit inputs", manifest["inputs"], ROOT),
    ("original XDec artifacts", json.loads((RUN / "original_xdec_sha256.json").read_text()), ROOT),
    ("sensitivity outputs", json.loads((RUN / "output_sha256.json").read_text()), RUN),
]:
    failures = [relative for relative, expected in hashes.items()
                if not (base / relative).is_file() or sha256(base / relative) != expected]
    assert not failures, f"Changed or missing {label}: {failures}"
    passed(f"Hashes match for {len(hashes)} {label}")
print("Recorded XDec commit:", manifest["xdec_commit"])
print("Recorded analysis packages:", manifest["packages"])

PASS: Hashes match for 15 explicit inputs
PASS: Hashes match for 88 original XDec artifacts
PASS: Hashes match for 128 sensitivity outputs
Recorded XDec commit: 66ee6bcf6b78d840837a6732de02fa3e620e669f
Recorded analysis packages: {'numpy': '2.5.4', 'pandas': '3.1.0', 'scipy': '1.18.1', 'statsmodels': '0.15.0', 'matplotlib': '3.11.2'}


### 3. Validate expression, proportions, fixed groups, and Stage-2 output shapes

Confirm identical gene/sample ordering between the TPM and count matrices, finite nonnegative values, unique identifiers, proportion sums, and tumor-only disjoint high/low groups. The saved groups must match the originals exactly. Stage 2 removes globally zero-expression genes; its output rows must match that rule.

In [3]:
tpm = pd.read_csv(PROGRESS / "data/processed/TCGA-PAAD_TPM_raw.tsv", sep="\t", index_col=0)
counts = pd.read_csv(PROGRESS / "data/processed/TCGA-PAAD_counts_raw.tsv", sep="\t", index_col=0)
assert counts.index.equals(tpm.index) and counts.columns.equals(tpm.columns)
for name, matrix in [("TPM", tpm), ("counts", counts)]:
    assert matrix.index.is_unique and matrix.columns.is_unique
    assert np.isfinite(matrix.to_numpy()).all() and matrix.to_numpy().min() >= 0
    print(name, matrix.shape)
props = pd.read_csv(ORIGINAL / "stage1_with_refs_k8/proportions.tsv", sep="\t", index_col=0)
assert props.index.is_unique and set(props.index) == set(tpm.columns)
assert np.isfinite(props.to_numpy()).all() and props.to_numpy().min() >= 0
assert np.allclose(props.sum(axis=1), 1)
active = tpm.index[tpm.sum(axis=1) > 0]
groups = {}
for profile in PROFILES:
    saved = pd.read_csv(RUN / f"{profile}_tertile_meta.tsv", sep="\t", index_col=0)
    original = pd.read_csv(ORIGINAL / f"tertile_meta/{profile}_tertile_meta.tsv", sep="\t", index_col=0)
    pd.testing.assert_frame_equal(saved, original)
    assert saved.index.is_unique and set(saved.group) == {"low", "high"}
    assert saved.index.isin(props.index).all()
    assert all(sample.split("-")[3][:2] == "01" for sample in saved.index)
    groups[profile] = saved
    print(profile, saved.group.value_counts().to_dict())
for folder in sorted((RUN / "tpm_stage2").iterdir()):
    means = pd.read_csv(folder / "stage2_means.tsv", sep="\t", index_col=0)
    errors = pd.read_csv(folder / "stage2_stderrs.tsv", sep="\t", index_col=0)
    assert means.index.equals(active)
    assert means.index.equals(errors.index) and means.columns.equals(errors.columns)
    assert means.columns.is_unique
    for matrix in [means, errors]:
        assert np.isfinite(matrix.to_numpy()).all() and matrix.to_numpy().min() >= 0
    print(folder.name, means.shape)
lib = counts.sum()
print("Raw library sizes (min / median / max):", lib.min(), lib.median(), lib.max())
passed("Input matrices, proportions, original groups, and all six TPM output pairs are valid")

TPM (59427, 182)
counts (59427, 182)
Profile_2 {'low': 60, 'high': 60}
Profile_5 {'high': 60, 'low': 60}
Profile_8 {'high': 60, 'low': 60}
main_k3 (54603, 3)
refs_k3 (54603, 3)
refs_k8 (54603, 8)
tertile_Profile_2 (54603, 16)
tertile_Profile_5 (54603, 16)
tertile_Profile_8 (54603, 16)
Raw library sizes (min / median / max): 19381044 46949950.0 85532339
PASS: Input matrices, proportions, original groups, and all six TPM output pairs are valid


### 4. Independently verify 36 NNLS estimates

For six genes in each of the two groups under each of the three profile groupings, solve the same least-squares problem using SciPy's independent NNLS solver. Compare the entire vector of eight cell-type estimates with XDec's vectorized coordinate-descent output (relative tolerance 1e-5; absolute tolerance 1e-4). This checks selected numerical examples rather than every gene or all uncertainty estimates.

In [4]:
solver_rows = []
for profile in PROFILES:
    means = pd.read_csv(RUN / "tpm_stage2" / f"tertile_{profile}" / "stage2_means.tsv", sep="\t", index_col=0)
    for group in ["low", "high"]:
        samples = groups[profile].index[groups[profile].group == group]
        design = props.loc[samples].to_numpy()
        for gene in ["CCR4", "CD28", "ICOS", "PTPRC", "GAPDH", "ACTB"]:
            expected, _ = nnls(design, tpm.loc[gene, samples].to_numpy())
            actual = means.loc[gene, [f"{p}_{group}" for p in props.columns]].to_numpy()
            difference = float(np.max(np.abs(expected - actual)))
            assert np.allclose(expected, actual, rtol=1e-5, atol=1e-4), (profile, group, gene, difference)
            solver_rows.append({"profile_grouping": profile, "group": group, "gene": gene,
                                "max_abs_difference_vs_scipy_nnls": difference})
solver_checks = pd.DataFrame(solver_rows)
saved_solver = pd.read_csv(RUN / "independent_solver_validation.tsv", sep="\t")
pd.testing.assert_frame_equal(solver_checks, saved_solver, check_exact=False, rtol=1e-5, atol=1e-4)
print(solver_checks.groupby("profile_grouping").max_abs_difference_vs_scipy_nnls.max().to_string())
passed(f"{len(solver_checks)} independent NNLS solutions match XDec and saved validation")

profile_grouping
Profile_2    4.365575e-11
Profile_5    2.090019e-09
Profile_8    4.949641e-11
PASS: 36 independent NNLS solutions match XDec and saved validation


### 5. Recompute all gene-level tests, BH corrections, and summary counts

Use the same conditional normal z-test as the sensitivity analysis to isolate input-unit effects. BH correction is performed over every finite test separately for each input/profile, before filtering. Genes with zero standard error remain explicitly untested. The original fivefold rule uses `log2((mean_high + 1)/(mean_low + 1))`; the pseudocount has different meanings in raw-count and TPM units.

In [5]:
summary_rows = []
recomputed = {}
for profile in PROFILES:
    for unit, folder in [
        ("raw_counts", ORIGINAL / f"stage2_tertile_{profile}"),
        ("TPM", RUN / "tpm_stage2" / f"tertile_{profile}"),
    ]:
        means = pd.read_csv(folder / "stage2_means.tsv", sep="\t", index_col=0)
        errors = pd.read_csv(folder / "stage2_stderrs.tsv", sep="\t", index_col=0)
        high, low = means[f"{profile}_high"], means[f"{profile}_low"]
        se = np.hypot(errors[f"{profile}_high"], errors[f"{profile}_low"])
        valid = se > 0
        z = pd.Series(np.nan, index=means.index)
        z.loc[valid] = (high[valid] - low[valid]) / se[valid]
        p = pd.Series(np.nan, index=means.index)
        p.loc[valid] = 2 * norm.sf(abs(z[valid]))
        q = pd.Series(np.nan, index=means.index)
        q.loc[valid] = multipletests(p[valid], method="fdr_bh")[1]
        fc = np.log2((high + 1) / (low + 1))
        saved = pd.read_csv(RUN / "differential_expression" / unit / f"{profile}_all_genes.tsv", sep="\t", index_col=0)
        assert saved.index.equals(means.index)
        for column, expected in [("z", z), ("pval", p), ("padj_BH", q), ("log2fc", fc)]:
            np.testing.assert_allclose(saved[column], expected, rtol=1e-10, atol=1e-14, equal_nan=True)
        assert saved.test_status.eq("tested").equals(valid)
        masks = {"FDR_significant": q < .05,
                 "FDR_and_5fold": (q < .05) & (abs(fc) >= np.log2(5)),
                 "nominal_and_5fold": (p < .05) & (abs(fc) >= np.log2(5))}
        for rule, mask in masks.items():
            assert np.array_equal(saved[rule], mask)
            selected = fc[mask]
            summary_rows.append({"profile": profile, "input": unit, "rule": rule,
                "tested_genes": int(valid.sum()), "total": len(selected),
                "up_high": int((selected > 0).sum()), "up_low": int((selected < 0).sum())})
        if unit == "raw_counts":
            original = pd.read_csv(ORIGINAL / f"{profile}_tertile_DE_genes.tsv", sep="\t", index_col=0)
            assert set(original.index) == set(means.index[masks["nominal_and_5fold"]])
        recomputed[(profile, unit)] = saved
summary = pd.DataFrame(summary_rows)
pd.testing.assert_frame_equal(summary, pd.read_csv(RUN / "DE_summary.tsv", sep="\t"))
print(summary.to_string(index=False))
passed("All six full test tables, BH corrections, 18 summary rows, and three original nominal gene sets reproduced")

  profile      input              rule  tested_genes  total  up_high  up_low
Profile_2 raw_counts   FDR_significant         53596      0        0       0
Profile_2 raw_counts     FDR_and_5fold         53596      0        0       0
Profile_2 raw_counts nominal_and_5fold         53596    206       10     196
Profile_2        TPM   FDR_significant         53596      0        0       0
Profile_2        TPM     FDR_and_5fold         53596      0        0       0
Profile_2        TPM nominal_and_5fold         53596    486        1     485
Profile_5 raw_counts   FDR_significant         53558     74       73       1
Profile_5 raw_counts     FDR_and_5fold         53558     37       36       1
Profile_5 raw_counts nominal_and_5fold         53558    584      276     308
Profile_5        TPM   FDR_significant         53558     16       16       0
Profile_5        TPM     FDR_and_5fold         53558      6        6       0
Profile_5        TPM nominal_and_5fold         53558    575      116     459

### 6. Verify gene overlap and enrichment completion

Check the gene overlap and effect directions directly from the complete test tables. Enrichment is not rerun: verify the successful retry has all 24 status records and that significant-term tables agree with recorded counts. The original failed KEGG attempt is preserved separately. Lists below five genes are explicitly skipped, not interpreted as negative enrichment results.

In [6]:
for profile in PROFILES:
    a, b = recomputed[(profile, "raw_counts")], recomputed[(profile, "TPM")]
    aset, bset = set(a.index[a.FDR_and_5fold]), set(b.index[b.FDR_and_5fold])
    shared = sorted(aset & bset)
    saved = pd.read_csv(RUN / "effect_concordance.tsv", sep="\t").set_index("profile").loc[profile]
    assert len(shared) == saved.FDR_5fold_shared
    assert len(aset | bset) == saved.FDR_5fold_union
    if shared:
        assert np.array_equal(np.sign(a.loc[shared].log2fc), np.sign(b.loc[shared].log2fc))
    print(profile, "raw:", len(aset), "TPM:", len(bset), "shared:", len(shared))
status = pd.read_csv(RUN / "enrichment_attempt_2/status.tsv", sep="\t")
assert len(status) == 24
assert not status.duplicated(["profile", "input", "direction", "analysis"]).any()
assert not status.status.eq("failed").any()
for row in status.itertuples():
    if row.status.startswith("success"):
        path = RUN / "enrichment_attempt_2" / f"{row.profile}_{row.input}_{row.direction}_{row.analysis}_FDR05.tsv"
        try:
            terms = pd.read_csv(path, sep="\t")
        except pd.errors.EmptyDataError:
            terms = pd.DataFrame()
        assert len(terms) == row.significant_terms
        if len(terms):
            assert terms["p.adjust"].lt(.05).all()
print(status[["profile", "input", "direction", "analysis", "status", "significant_terms"]].to_string(index=False))
print("\nPathway overlap:")
print(pd.read_csv(RUN / "pathway_overlap.tsv", sep="\t")[["profile", "analysis", "raw_significant", "TPM_significant", "shared"]].to_string(index=False))
passed("Gene overlap/directions and all 24 enrichment records verified")
print("\nAll", len(checks), "check groups passed.")

Profile_2 raw: 0 TPM: 0 shared: 0
Profile_5 raw: 37 TPM: 6 shared: 6
Profile_8 raw: 50 TPM: 17 shared: 17
  profile      input direction analysis                       status  significant_terms
Profile_2 raw_counts  UpInHigh    GO_BP   not_run_fewer_than_5_genes                  0
Profile_2 raw_counts  UpInHigh     KEGG   not_run_fewer_than_5_genes                  0
Profile_2 raw_counts   UpInLow    GO_BP   not_run_fewer_than_5_genes                  0
Profile_2 raw_counts   UpInLow     KEGG   not_run_fewer_than_5_genes                  0
Profile_2        TPM  UpInHigh    GO_BP   not_run_fewer_than_5_genes                  0
Profile_2        TPM  UpInHigh     KEGG   not_run_fewer_than_5_genes                  0
Profile_2        TPM   UpInLow    GO_BP   not_run_fewer_than_5_genes                  0
Profile_2        TPM   UpInLow     KEGG   not_run_fewer_than_5_genes                  0
Profile_5 raw_counts  UpInHigh    GO_BP                      success                173
Profile_5 raw_

### Interpretation

* The computational checks establish consistency between the saved files, recomputed tests, and selected independent solver results. All original files in the integrity manifests remain unchanged.
* The lymphocyte signal persists across input units: the six TPM genes passing BH plus the effect filter are shared with raw counts, and four significant KEGG pathways overlap. Exact DE lists differ substantially.
* The CAF profile has no genes passing BH in either input. Enteroendocrine enrichment is not significant with TPM under this recipe.
* These are conditional, exploratory analyses. Neither the z-tests nor their BH correction account for Stage-1 estimation/group-selection uncertainty. The comparison does not identify the uniquely correct Stage-2 input unit or provide independent biological validation.
* See the saved run's `SUMMARY.md`, `manifest.json`, complete test tables and annotation snapshot for interpretation and provenance. Existing analysis artifacts were not modified by this notebook.
